## Cluster Analysis: Subgroups within Silent Middle

This section performs exploratory clustering analysis on the silent middle respondents to identify distinct behavioral subgroups.

**Important:** The response pattern features (`resp_*`) are derived from choice task responses (the same columns used to define the silent middle target). Using them here is valid because:
- We are analyzing characteristics of already-identified silent middle respondents
- We are NOT using them to predict silent middle membership (which would be data leakage)

The goal is to understand heterogeneity within the silent middle group, not to predict it.

In [ ]:
# Filter to silent middle respondents only
df_sm = df[df["silent_middle"] == 1].copy()

# Features for clustering
cluster_features = [
    "motivationamount", 
    "logmotivationlength", 
    "resp_personal_std",
    "resp_study_corr",
    "grade",
    "log_completion_time"
]

# Keep only features that exist
cluster_features = [c for c in cluster_features if c in df_sm.columns]
print(f"Clustering features: {cluster_features}")

# Prepare data (drop rows with missing values)
X_cluster = df_sm[cluster_features].dropna()
print(f"\nClustering {len(X_cluster):,} silent middle respondents")
print(f"Dropped {len(df_sm) - len(X_cluster):,} with missing values")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# K-means with 4 clusters
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Add cluster labels
df_clustered = X_cluster.copy()
df_clustered["cluster"] = clusters

print("Cluster distribution:")
print(df_clustered["cluster"].value_counts().sort_index())

In [ ]:
# Analyze cluster profiles
print("Cluster Profiles (mean values):")
print("=" * 80)

cluster_profiles = df_clustered.groupby("cluster")[cluster_features].mean()
cluster_sizes = df_clustered["cluster"].value_counts().sort_index()
cluster_pcts = (cluster_sizes / len(df_clustered) * 100).round(1)

for c in range(n_clusters):
    profile = cluster_profiles.loc[c]
    n = cluster_sizes[c]
    pct = cluster_pcts[c]
    
    print(f"\nCluster {c}: n={n:,} ({pct}%)")
    print(f"  Motivation amount:     {profile['motivationamount']:.1f}")
    print(f"  Log motivation length: {profile['logmotivationlength']:.2f}")
    print(f"  Response std:          {profile['resp_personal_std']:.2f}")
    print(f"  Study correlation:     {profile['resp_study_corr']:.3f}")
    print(f"  Grade:                 {profile['grade']:.2f}")
    if 'log_completion_time' in cluster_features:
        print(f"  Log completion time:   {profile['log_completion_time']:.2f}")

In [ ]:
# Cluster names based on interpretation
cluster_names = {
    0: "Cluster 0:\nHigh Variance Engaged",
    1: "Cluster 1:\nConformist Engaged", 
    2: "Cluster 2:\nLow Engagement Passive",
    3: "Cluster 3:\nIndependent Moderate"
}

# Normalize cluster profiles to show relative contribution (0-1 scale per feature)
profile_normalized = cluster_profiles.copy()
for col in cluster_features:
    col_min = profile_normalized[col].min()
    col_max = profile_normalized[col].max()
    if col_max > col_min:
        profile_normalized[col] = (profile_normalized[col] - col_min) / (col_max - col_min)
    else:
        profile_normalized[col] = 0.5

# Feature display names and colors
feature_labels = {
    "motivationamount": "Motivation Count",
    "logmotivationlength": "Text Length (log)",
    "resp_personal_std": "Response Variance",
    "resp_study_corr": "Conformity to Group",
    "grade": "Satisfaction",
    "log_completion_time": "Completion Time (log)"
}

feature_colors = {
    "motivationamount": "#1f77b4",
    "logmotivationlength": "#ff7f0e",
    "resp_personal_std": "#2ca02c",
    "resp_study_corr": "#d62728",
    "grade": "#9467bd",
    "log_completion_time": "#8c564b"
}

# Create 4 subplots - one per cluster, features ranked high to low
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

n_features = len(cluster_features)

for c in range(n_clusters):
    ax = axes[c]
    
    # Get values for this cluster and sort by descending order
    values = profile_normalized.loc[c, cluster_features].values
    feature_order = np.argsort(values)[::-1]  # Sort high to low
    
    sorted_values = values[feature_order]
    sorted_features = [cluster_features[i] for i in feature_order]
    sorted_labels = [feature_labels[f] for f in sorted_features]
    sorted_colors = [feature_colors[f] for f in sorted_features]
    
    # Plot bars
    bars = ax.bar(range(n_features), sorted_values, color=sorted_colors, edgecolor="white", linewidth=0.5)
    
    ax.set_xticks(range(n_features))
    ax.set_xticklabels(sorted_labels, fontsize=8, rotation=45, ha="right")
    ax.set_ylabel("Normalized Value" if c == 0 else "")
    ax.set_title(f"{cluster_names[c]}\n(n={cluster_sizes[c]:,}, {cluster_pcts[c]}%)", fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.15)
    
    # Add value labels on bars
    for bar, val in zip(bars, sorted_values):
        ax.annotate(
            f"{val:.2f}",
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02),
            ha="center",
            fontsize=8,
            color="gray"
        )

# Add legend for features
legend_elements = [plt.Rectangle((0,0), 1, 1, facecolor=feature_colors[f], label=feature_labels[f]) 
                   for f in cluster_features]
fig.legend(handles=legend_elements, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.08), fontsize=9)

plt.suptitle("Cluster Profiles: Features Ranked High to Low per Cluster", fontsize=14, fontweight="bold", y=1.18)
plt.tight_layout()
plt.show()

In [ ]:
# Alternative: Z-score normalization (shows how many std deviations from mean)
profile_zscore = cluster_profiles.copy()
for col in cluster_features:
    col_mean = profile_zscore[col].mean()
    col_std = profile_zscore[col].std()
    if col_std > 0:
        profile_zscore[col] = (profile_zscore[col] - col_mean) / col_std
    else:
        profile_zscore[col] = 0

# Fixed y-axis limits
y_min = -1.5
y_max = 1.5

# Create 4 subplots - one per cluster, features ranked high to low
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=True)

for c in range(n_clusters):
    ax = axes[c]
    
    # Get values for this cluster and sort by descending order
    values = profile_zscore.loc[c, cluster_features].values
    feature_order = np.argsort(values)[::-1]  # Sort high to low
    
    sorted_values = values[feature_order]
    sorted_features = [cluster_features[i] for i in feature_order]
    sorted_labels = [feature_labels[f] for f in sorted_features]
    sorted_colors = [feature_colors[f] for f in sorted_features]
    
    # Plot bars (can be negative with z-score)
    bars = ax.bar(range(n_features), sorted_values, color=sorted_colors, edgecolor="white", linewidth=0.5)
    
    ax.set_xticks(range(n_features))
    ax.set_xticklabels(sorted_labels, fontsize=8, rotation=45, ha="right")
    ax.set_ylabel("Z-score" if c == 0 else "")
    ax.set_title(f"{cluster_names[c]}\n(n={cluster_sizes[c]:,}, {cluster_pcts[c]}%)", fontsize=10, fontweight="bold")
    ax.set_ylim(y_min, y_max)
    ax.axhline(y=0, color="black", linestyle="-", linewidth=0.5, alpha=0.5)
    
    # Add value labels on bars
    for bar, val in zip(bars, sorted_values):
        y_pos = val + 0.1 if val >= 0 else val - 0.25
        ax.annotate(
            f"{val:.2f}",
            xy=(bar.get_x() + bar.get_width() / 2, y_pos),
            ha="center",
            fontsize=8,
            color="gray"
        )

# Add legend for features
legend_elements = [plt.Rectangle((0,0), 1, 1, facecolor=feature_colors[f], label=feature_labels[f]) 
                   for f in cluster_features]
fig.legend(handles=legend_elements, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.08), fontsize=9)

plt.suptitle("Cluster Profiles (Z-score): Above/Below Average per Feature", fontsize=14, fontweight="bold", y=1.18)
plt.tight_layout()
plt.show()

print("Interpretation: Positive = above average across clusters, Negative = below average")

### Demographic Profiling

Clusters were formed based on behavioral features only. Now we examine whether clusters differ demographically (age, gender, education).

In [ ]:
# Demographic profiling of clusters
df_clustered_demo = df_sm.loc[X_cluster.index].copy()
df_clustered_demo["cluster"] = clusters

demo_cols = ["age", "gender", "education"]
demo_cols = [c for c in demo_cols if c in df_clustered_demo.columns]

print("Demographic Profile by Cluster")
print("=" * 80)

# Create one figure with 3 subplots (one per demographic)
fig, axes = plt.subplots(1, len(demo_cols), figsize=(18, 5))

for idx, col in enumerate(demo_cols):
    ax = axes[idx]
    
    # Cross-tabulation with percentages
    ct = pd.crosstab(df_clustered_demo["cluster"], df_clustered_demo[col], normalize="index") * 100
    print(f"\n{col.upper()} distribution:")
    print(ct.round(1))
    
    # Plot grouped bar chart
    ct.plot(kind="bar", ax=ax, width=0.8)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Percentage (%)")
    ax.set_title(f"{col.title()} Distribution by Cluster", fontsize=12, fontweight="bold")
    ax.legend(title=col.title(), fontsize=8)
    ax.set_xticklabels([f"Cluster {i}" for i in range(n_clusters)], rotation=0)
    ax.set_ylim(0, 100)

plt.suptitle("Demographic Profiles by Cluster", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Summary table with all demographics
print("\n" + "=" * 80)
print("Summary: Dominant demographic per cluster")
print("=" * 80)
for c in range(n_clusters):
    cluster_data = df_clustered_demo[df_clustered_demo["cluster"] == c]
    print(f"\nCluster {c} (n={len(cluster_data):,}):")
    for col in demo_cols:
        mode_val = cluster_data[col].mode()
        if len(mode_val) > 0:
            mode_pct = (cluster_data[col] == mode_val.iloc[0]).mean() * 100
            print(f"  {col}: {mode_val.iloc[0]} ({mode_pct:.1f}%)")

### Cluster Interpretation

Based on the cluster profiles, we identify four distinct subgroups within the silent middle:

| Cluster | Name | Characteristics |
|---------|------|----------------|
| 0 | **High Variance Engaged** | High response variance, engaged with motivations, moderate conformity |
| 1 | **Conformist Engaged** | Highest motivation count, strongly conforms to group median, satisfied |
| 2 | **Low Engagement Passive** | Minimal motivations, low text, quick completion time |
| 3 | **Independent Moderate** | Low conformity to group, consistent responses, highest satisfaction |

**Key Insights:**

1. **Engagement varies widely** - From highly engaged writers (Cluster 1: ~14 motivations) to near-silent participants (Cluster 2: ~1 motivation)

2. **Conformity is independent of engagement** - Cluster 1 writes extensively AND follows the group; Cluster 3 is moderate but independent

3. **Satisfaction correlates with independence** - The most satisfied cluster (3) is also the least conformist

4. **The "silent" middle is not silent** - Most of the silent middle actively provide motivations

This heterogeneity explains why predicting silent middle membership is difficult - the group is defined by outcome (moderate preferences) but arrives there through different behavioral paths.